In [2]:
!cp /data/ABC_Dataset/ABC_Dataset/PDMX_MXL_abci/mxl_abci/0/0/Qma1n8iRYvkpWPafjrHowRo2xXdYin8i632a9o2UxLz1Qc.abci /home/ubuntu/keshav/text2score/text2music/notebooks/test.abci

In [1]:
from text2music.preprocess.abci2abc import convert_abci_list_to_abc

# filepath = "/home/ubuntu/keshav/text2score/text2music/notebooks/test.abci"
filepath = "/data/ABC_Dataset/ABC_Dataset/PDMX_MXL_abci/mxl_abci/5/57/QmfZ535U7zXdmqnDRJuKhKXrzgdBh3V6DU7qDZkcDqMj4B.abci"
convert_abci_list_to_abc([filepath])

In [2]:
from text2music.preprocess.abc2xml import convert_abc_list_to_xml

abc_filepath = filepath.replace('.abci', '.abc')
convert_abc_list_to_xml([abc_filepath], abc2xml_script='../data/utils/abc_helper/abc2xml.py')

-- decoded from utf-8
-- skipped header: (field X,1)
-- done in 0.50 secs


In [3]:
from text2music.preprocess.xml2plan import convert_xml_list_to_plan_pkl

xml_filepath = abc_filepath.replace('.abc', '.xml')
convert_xml_list_to_plan_pkl([xml_filepath])

/data/conda/envs/text2score/lib/python3.11/site-packages/partitura/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
import partitura as pt
import xml.etree.ElementTree as ET
from typing import Dict, Optional, List, Set, Union
import numpy as np
import warnings

# Ignore user warnings
warnings.filterwarnings("ignore", category=UserWarning)

# --- HELPER FUNCTION FOR NAN HANDLING ---
def safe_int_or_none(value) -> Optional[int]:
    """
    Safely converts a value to int. 
    Returns None if value is NaN, None, or invalid.
    """
    try:
        if value is None:
            return None
        # Check for NaN (float('nan') != float('nan'))
        if isinstance(value, float) and value != value:
            return None
        return int(value)
    except (ValueError, TypeError):
        return None

# --- TEMPO EXTRACTION ---
def find_tempo_in_measure(measure_element: ET.Element) -> Optional[int]:
    """Checks for tempo markings within a single measure element."""
    for direction in measure_element.findall('direction'):
        sound_element = direction.find('sound')
        if sound_element is not None and 'tempo' in sound_element.attrib:
            try:
                return int(float(sound_element.attrib['tempo']))
            except (ValueError, TypeError):
                continue 

        direction_type = direction.find('direction-type')
        if direction_type is not None:
            metronome = direction_type.find('metronome')
            if metronome is not None:
                per_minute = metronome.find('per-minute')
                if per_minute is not None and per_minute.text:
                    try:
                        return int(per_minute.text)
                    except (ValueError, TypeError):
                        continue 
    return None

def extract_tempos(filepath: str) -> Dict[int, Optional[int]]:
    """Parses a MusicXML file and extracts tempo for each measure."""
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()
    except ET.ParseError as e:
        print(f"Error parsing XML file: {e}")
        return {}

    tempo_by_measure: Dict[int, Optional[int]] = {}
    current_tempo: Optional[int] = None

    first_part = root.find('part')
    if first_part is None:
        return {}

    measures = first_part.findall('measure')
    for measure_element in measures:
        measure_number_str = measure_element.get('number')
        if not measure_number_str:
            continue
        
        measure_number = int(measure_number_str)
        found_tempo = find_tempo_in_measure(measure_element)
        if found_tempo is not None:
            current_tempo = found_tempo
        
        tempo_by_measure[measure_number] = current_tempo
        
    return tempo_by_measure


# --- XML INFO EXTRACTION (STEP 1) ---
def get_raw_xml_info(musicxml_file: str, verbose: bool = False) -> list:
    """
    Extracts raw XML information. 
    Uses safe_int_or_none to prevent crashes on NaN.
    """
    score = pt.load_musicxml(musicxml_file)
    all_tempos = extract_tempos(musicxml_file)
    data = []

    for i, part in enumerate(score.parts):
        part_name = part.part_name
        if not part_name:
            part_name = f"Piano {i+1}" if i > 0 else "Piano"

        # Unique part naming
        for p in data:
            n = 1
            if p['part_name'] == part_name:
                part_name += f" {n+1}"
                while any(p['part_name'] == part_name for p in data):
                    n += 1
                    part_name = f"{part.part_name} {n+1}"
        
        if verbose:
            print(f"Part Name: {part_name}")

        extended_score_note_array = pt.utils.music.ensure_notearray(
            part,
            include_pitch_spelling=True,
            include_key_signature=True,
            include_time_signature=True,
            include_metrical_position=True,
            include_grace_notes=True
        )

        measures = list(part.iter_all(pt.score.Measure, include_subclasses=True))
        part_dynamics = list(part.dynamics)

        part_data = {"part_name": part_name, "measures": []}

        for measure in measures:
            # SAFETY: Using safe_int_or_none here prevents NaNs from propagating
            ts_num = safe_int_or_none(part.time_signature_map(measure.start.t)[0])
            ts_den = safe_int_or_none(part.time_signature_map(measure.start.t)[1])
            ts_beats = safe_int_or_none(part.time_signature_map(measure.start.t)[2])
            ks_fifths = safe_int_or_none(part.key_signature_map(measure.start.t)[0])
            ks_mode = safe_int_or_none(part.key_signature_map(measure.start.t)[1])

            # Extract Dynamics for this measure
            dynamics_in_measure = []
            for dynamic in part_dynamics:
                if measure.start.t <= dynamic.start.t < measure.end.t:
                    dynamics_in_measure.append({
                        "text": dynamic.text,
                        "start_time": dynamic.start.t,
                        "rel_onset_div": dynamic.start.t - measure.start.t
                    })

            measure_data = {
                "measure": measure.number,
                "start_time": measure.start.t,
                "end_time": measure.end.t,
                "ts_num": ts_num,
                "ts_den": ts_den,
                "ts_beats": ts_beats,
                "ks_fifths": ks_fifths,
                "ks_mode": ks_mode,
                "tempo": all_tempos.get(measure.number, None),
                "dynamics": dynamics_in_measure
            }
            
            notes_in_measure = [n for n in extended_score_note_array if measure.start.t <= n[4] < measure.end.t]
            notes = []
            for n, note in enumerate(notes_in_measure):
                note_data = {
                    "pitch": safe_int_or_none(note['pitch']), # Safety check
                    "start_time": note['onset_div'],
                    "end_time": note['onset_div'] + note['duration_div'],
                    "rel_onset_div": note['rel_onset_div'],
                    "tot_measure_div": note['tot_measure_div'],
                    "duration": note['duration_div'],
                    "is_grace": note['is_grace'],
                    "grace_type": note['grace_type']
                }
                notes.append(note_data)
            measure_data["notes"] = notes
            part_data["measures"].append(measure_data)
        data.append(part_data)

    return data


# --- DATA COMBINATION (STEP 2) ---
def combine_raw_xml_info(data):
    """
    Combines raw XML info.
    Aggregates dynamics per measure.
    """
    # 1. Pre-calculate aggregated dynamics
    aggregated_dynamics = {}
    for part_data in data:
        for measure_data in part_data['measures']:
            measure_num = measure_data['measure']
            if measure_num not in aggregated_dynamics:
                aggregated_dynamics[measure_num] = []
            part_dyns = measure_data.get('dynamics', [])
            aggregated_dynamics[measure_num].extend(part_dyns)

    # 2. Sort dynamics chronologically
    for measure_num in aggregated_dynamics:
        aggregated_dynamics[measure_num].sort(key=lambda x: x['start_time'])

    # 3. Build combined structure
    combined_measures = {}
    for part_data in data:
        part_name = part_data['part_name']
        for measure_data in part_data['measures']:
            measure_num = measure_data['measure']
            if f"Measure: {measure_num}" not in combined_measures:
                combined_measures[f"Measure: {measure_num}"] = {}
            
            measure_aggregated_dyns = aggregated_dynamics.get(measure_num, [])

            combined_measures[f"Measure: {measure_num}"][part_name] = {
                'info': {
                    'start_time': measure_data['start_time'],
                    'end_time': measure_data['end_time'],
                    'ts_num': measure_data['ts_num'],
                    'ts_den': measure_data['ts_den'],
                    'ts_beats': measure_data['ts_beats'],
                    'ks_fifths': measure_data['ks_fifths'],
                    'ks_mode': measure_data['ks_mode'],
                    'tempo': measure_data['tempo'],
                    'notes': measure_data['notes'],
                    'dynamics': measure_aggregated_dyns
                }
            }
    return combined_measures


# --- PLAN GENERATION (STEP 3) ---
def get_full_plan(combined_data):
    """
    Generates full plan.
    - Handles NaN/None safely.
    - Filters chords (>=2 notes).
    - Includes aggregated dynamics.
    """
    plan = {}
    all_instruments = set()
    
    for measure in combined_data.keys():
        measure_info = combined_data[measure]
        plan[measure] = {}

        for instrument in measure_info.keys():
            all_instruments.add(instrument)

        plan[measure]['instruments'] = [
            inst for inst in combined_data[measure].keys() 
            if len(combined_data[measure][inst]['info']['notes']) > 0
        ]
        
        # --- 1. PITCH RANGE ---
        pitch_ranges = []
        for instrument in plan[measure]['instruments']:
            notes = combined_data[measure][instrument]['info']['notes']
            if notes:
                # Filter out None pitches
                pitches = [safe_int_or_none(n['pitch']) for n in notes]
                pitches = [p for p in pitches if p is not None]
                if pitches:
                    pitch_ranges.append((min(pitches), max(pitches)))
        
        if pitch_ranges:
            plan[measure]['pitch_range'] = {
                'min': min(p[0] for p in pitch_ranges),
                'max': max(p[1] for p in pitch_ranges)
            }
        else:
            plan[measure]['pitch_range'] = {'min': None, 'max': None}
        
        # --- 2. NOTE DENSITY ---
        total_notes = sum(len(combined_data[measure][inst]['info']['notes']) for inst in plan[measure]['instruments'])
        if plan[measure]['instruments']:
            plan[measure]['note_density'] = float(round(total_notes / len(plan[measure]['instruments']), 2))
        else:
            plan[measure]['note_density'] = 0.0
        
        # --- 3. TIME SIGNATURE ---
        plan[measure]['time_signature'] = None
        if plan[measure]['instruments']:
            first_inst = plan[measure]['instruments'][0]
            # Retrieve potentially None values
            ts_num = safe_int_or_none(combined_data[measure][first_inst]['info']['ts_num'])
            ts_den = safe_int_or_none(combined_data[measure][first_inst]['info']['ts_den'])
            
            if ts_num is not None and ts_den is not None:
                plan[measure]['time_signature'] = f"{ts_num}/{ts_den}" 
        
        # --- 4. KEY SIGNATURE ---
        instruments_to_exclude = ['Drum', 'Percussion', 'Drumset', 'Drums', 'Saxophone', 'Clarinet']
        plan[measure]['key_signature'] = None
        
        if 'Piano' in plan[measure]['instruments']:
            plan[measure]['key_signature'] = safe_int_or_none(combined_data[measure]['Piano']['info']['ks_fifths'])
        elif plan[measure]['instruments']:
            key_signatures = [
                safe_int_or_none(combined_data[measure][inst]['info']['ks_fifths'])
                for inst in plan[measure]['instruments'] 
                if not any(excluded in inst for excluded in instruments_to_exclude)
            ]
            # Filter Nones
            key_signatures = [k for k in key_signatures if k is not None]
            if key_signatures:
                plan[measure]['key_signature'] = max(set(key_signatures), key=key_signatures.count)

        # --- 5. KEY MODE ---
        first_inst = plan[measure]['instruments'][0] if plan[measure]['instruments'] else None
        if first_inst and 'info' in combined_data[measure][first_inst]:
            plan[measure]['key_mode'] = safe_int_or_none(combined_data[measure][first_inst]['info']['ks_mode'])
        else:
            plan[measure]['key_mode'] = None

        # --- 6. TEMPO ---
        if first_inst and 'info' in combined_data[measure][first_inst]:
            plan[measure]['tempo'] = safe_int_or_none(combined_data[measure][first_inst]['info']['tempo'])
        else:
            plan[measure]['tempo'] = None

        # --- 7. DYNAMICS ---
        plan[measure]['dynamics'] = []
        if first_inst and 'info' in combined_data[measure][first_inst]:
            raw_dynamics = combined_data[measure][first_inst]['info'].get('dynamics', [])
            clean_dynamics = []
            for d in raw_dynamics:
                onset = safe_int_or_none(d['rel_onset_div'])
                if onset is not None:
                    clean_dynamics.append({'onset': onset, 'value': str(d['text'])})
            plan[measure]['dynamics'] = clean_dynamics

        # --- 8. CHORDS ---
        plan[measure]['chords'] = []
        if not plan[measure]['instruments']:
            continue  

        onset_groups = {}
        for instrument in plan[measure]['instruments']:
            notes = combined_data[measure][instrument]['info']['notes']
            for note in notes:
                if note['is_grace'] == 0:
                    onset = note['rel_onset_div']
                    raw_pitch = safe_int_or_none(note['pitch'])
                    
                    if raw_pitch is None: 
                        continue
                        
                    note_pitch = int(raw_pitch % 12)
                    if onset not in onset_groups:
                        onset_groups[onset] = set()
                    onset_groups[onset].add(note_pitch)

        unique_chords = []
        seen_chords = set()
        sorted_onsets = sorted(onset_groups.keys())

        for onset in sorted_onsets:
            current_chord_set = onset_groups[onset]
            if len(current_chord_set) < 2:
                continue
            
            chord_signature = tuple(sorted(current_chord_set))
            if chord_signature not in seen_chords:
                seen_chords.add(chord_signature)
                unique_chords.append(current_chord_set)

        plan[measure]['chords'] = unique_chords

    plan['all_instruments'] = list(all_instruments)
    return plan


# --- MAIN PIPELINE ---
def get_full_plan_pipeline(musicxml_file: str):
    """Pipeline to get the full plan from a MusicXML file."""
    data = get_raw_xml_info(musicxml_file)
    combined_data = combine_raw_xml_info(data)
    full_plan = get_full_plan(combined_data)
    return full_plan

filepath = "/data/ABC_Dataset/ABC_Dataset/PDMX_MXL_abci/mxl_abci/5/57/QmfZ535U7zXdmqnDRJuKhKXrzgdBh3V6DU7qDZkcDqMj4B.xml"
get_full_plan_pipeline(filepath)

--- Part 1: Piano ---
Dynamic: p, Start Time: 0, End Time: TimePoint t=750 quarter=30
Dynamic: crescendo, Start Time: 30, End Time: TimePoint t=150 quarter=30
Dynamic: diminuendo, Start Time: 195, End Time: TimePoint t=240 quarter=30
Dynamic: crescendo, Start Time: 270, End Time: TimePoint t=390 quarter=30
Dynamic: diminuendo, Start Time: 435, End Time: TimePoint t=480 quarter=30
Dynamic: diminuendo, Start Time: 510, End Time: TimePoint t=600 quarter=30
Dynamic: sf, Start Time: 510, End Time: None
Dynamic: diminuendo, Start Time: 630, End Time: TimePoint t=720 quarter=30
Dynamic: sf, Start Time: 630, End Time: None
Dynamic: ff, Start Time: 750, End Time: TimePoint t=870 quarter=30
Dynamic: diminuendo, Start Time: 810, End Time: TimePoint t=870 quarter=30
Dynamic: p, Start Time: 870, End Time: TimePoint t=960 quarter=30
Dynamic: p, Start Time: 960, End Time: TimePoint t=2100 quarter=30
Dynamic: crescendo, Start Time: 990, End Time: TimePoint t=1110 quarter=30
Dynamic: diminuendo, Start 

In [1]:
from text2music.data.utils.xml2plan import *

# xml_filepath = abc_filepath.replace('.abc', '.xml')
xml_filepath = "/home/ubuntu/keshav/text2score/text2music/notebooks/xml_score.xml"

data = get_raw_xml_info(xml_filepath)
combined_data = combine_raw_xml_info(data)
full_plan = get_full_plan(combined_data)
full_plan

/data/conda/envs/text2score/lib/python3.11/site-packages/partitura/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


{'Measure: 1': {'instruments': ['Piano'],
  'pitch_range': {'min': 60, 'max': 60},
  'note_density': 1.0,
  'time_signature': '2/2',
  'key_signature': -4,
  'key_mode': 1,
  'tempo': 232,
  'chords': []},
 'Measure: 2': {'instruments': ['Piano'],
  'pitch_range': {'min': 65, 'max': 77},
  'note_density': 4.0,
  'time_signature': '2/2',
  'key_signature': -4,
  'key_mode': 1,
  'tempo': 232,
  'chords': []},
 'Measure: 3': {'instruments': ['Piano'],
  'pitch_range': {'min': 53, 'max': 80},
  'note_density': 14.0,
  'time_signature': '2/2',
  'key_signature': -4,
  'key_mode': 1,
  'tempo': 232,
  'chords': [{0, 5, 8}]},
 'Measure: 4': {'instruments': ['Piano'],
  'pitch_range': {'min': 52, 'max': 79},
  'note_density': 8.0,
  'time_signature': '2/2',
  'key_signature': -4,
  'key_mode': 1,
  'tempo': 232,
  'chords': [{0, 4, 7, 10}]},
 'Measure: 5': {'instruments': ['Piano'],
  'pitch_range': {'min': 52, 'max': 82},
  'note_density': 17.0,
  'time_signature': '2/2',
  'key_signature': 

In [11]:
def combine_raw_xml_info(data):
    """
    Combines the raw XML information from multiple parts into a single dictionary.
    Includes an aggregated list of dynamics for each measure across all parts.

    Args:
        data: A list of dictionaries containing raw XML information for each part.

    Returns:
        A combined dictionary with all parts' data.
    """
    
    # 1. Pre-calculate the aggregated dynamics per measure
    aggregated_dynamics = {}
    
    for part_data in data:
        for measure_data in part_data['measures']:
            measure_num = measure_data['measure']
            if measure_num not in aggregated_dynamics:
                aggregated_dynamics[measure_num] = []
            
            # Collect dynamics from this part if they exist
            # (Assumes get_raw_xml_info has been updated to include 'dynamics')
            part_dyns = measure_data.get('dynamics', [])
            aggregated_dynamics[measure_num].extend(part_dyns)

    # 2. Sort the aggregated dynamics chronologically for each measure
    for measure_num in aggregated_dynamics:
        # Sort by global start_time to ensure chronological order across parts
        aggregated_dynamics[measure_num].sort(key=lambda x: x['start_time'])

    # 3. Build the combined structure
    combined_measures = {}
    for part_data in data:
        part_name = part_data['part_name']
        for measure_data in part_data['measures']:
            measure_num = measure_data['measure']
            if f"Measure: {measure_num}" not in combined_measures:
                combined_measures[f"Measure: {measure_num}"] = {}
            
            # Retrieve the pre-calculated aggregated list for this measure
            # We attach this list to every part so it's accessible regardless of which instrument is queried
            measure_aggregated_dyns = aggregated_dynamics.get(measure_num, [])

            combined_measures[f"Measure: {measure_num}"][part_name] = {
                'info': {
                    'start_time': measure_data['start_time'],
                    'end_time': measure_data['end_time'],
                    'ts_num': measure_data['ts_num'],
                    'ts_den': measure_data['ts_den'],
                    'ts_beats': measure_data['ts_beats'],
                    'ks_fifths': measure_data['ks_fifths'],
                    'ks_mode': measure_data['ks_mode'],
                    'tempo': measure_data['tempo'],
                    'notes': measure_data['notes'],
                    'dynamics': measure_aggregated_dyns  # Added aggregated dynamics
                }
            }

    return combined_measures

combined_data = combine_raw_xml_info(data)
combined_data

{'Measure: 1': {'Piano': {'info': {'start_time': 0,
    'end_time': 30,
    'ts_num': 2,
    'ts_den': 2,
    'ts_beats': 2,
    'ks_fifths': -4,
    'ks_mode': 1,
    'tempo': 232,
    'notes': [{'pitch': np.int32(60),
      'start_time': np.int32(0),
      'end_time': np.int32(30),
      'rel_onset_div': np.int32(90),
      'tot_measure_div': np.int32(120),
      'duration': np.int32(30),
      'is_grace': np.int8(0),
      'grace_type': np.str_('')}],
    'dynamics': [{'text': 'p', 'start_time': 0, 'rel_onset_div': 0}]}}},
 'Measure: 2': {'Piano': {'info': {'start_time': 30,
    'end_time': 150,
    'ts_num': 2,
    'ts_den': 2,
    'ts_beats': 2,
    'ks_fifths': -4,
    'ks_mode': 1,
    'tempo': 232,
    'notes': [{'pitch': np.int32(65),
      'start_time': np.int32(30),
      'end_time': np.int32(60),
      'rel_onset_div': np.int32(0),
      'tot_measure_div': np.int32(120),
      'duration': np.int32(30),
      'is_grace': np.int8(0),
      'grace_type': np.str_('')},
     {'p

In [13]:
def get_full_plan(combined_data):
    """
    Generates a full plan from the combined data.
    - Filters chords to only include simultaneities of >= 2 notes.
    - Converts all NumPy types to standard Python int/float.
    - Includes aggregated dynamics sorted by time.
    """

    plan = {}
    all_instruments = set()
    for measure in combined_data.keys():
        measure_info = combined_data[measure]
        plan[measure] = {}

        # Add instruments in this measure to all_instruments
        for instrument in measure_info.keys():
            all_instruments.add(instrument)

        # Add instruments which have notes in this measure
        plan[measure]['instruments'] = [
            instrument for instrument in combined_data[measure].keys() 
            if len(combined_data[measure][instrument]['info']['notes']) > 0
        ]
        
        # --- 1. PITCH RANGE ---
        pitch_ranges = []
        for instrument in plan[measure]['instruments']:
            notes = combined_data[measure][instrument]['info']['notes']
            if notes:
                pitches = [int(note['pitch']) for note in notes]
                max_pitch = max(pitches)
                min_pitch = min(pitches)
                pitch_ranges.append((min_pitch, max_pitch))
        if pitch_ranges:
            plan[measure]['pitch_range'] = {
                'min': int(min(p[0] for p in pitch_ranges)),
                'max': int(max(p[1] for p in pitch_ranges))
            }
        else:
            plan[measure]['pitch_range'] = {'min': None, 'max': None}
        
        # --- 2. NOTE DENSITY ---
        total_notes = sum(len(combined_data[measure][instrument]['info']['notes']) for instrument in plan[measure]['instruments'])
        if plan[measure]['instruments']:
            average_notes = float(round(total_notes / len(plan[measure]['instruments']), 2))
        else:
            average_notes = 0.0
        plan[measure]['note_density'] = average_notes
        
        # --- 3. TIME SIGNATURE ---
        if plan[measure]['instruments']:
            first_instrument = plan[measure]['instruments'][0]
            ts_num = int(combined_data[measure][first_instrument]['info']['ts_num'])
            ts_den = int(combined_data[measure][first_instrument]['info']['ts_den'])
            plan[measure]['time_signature'] = f"{ts_num}/{ts_den}" 
        else:
            plan[measure]['time_signature'] = None
        
        # --- 4. KEY SIGNATURE ---
        instruments_to_exclude = ['Drum', 'Percussion', 'Drumset', 'Drums', 'Saxophone', 'Clarinet']
        plan[measure]['key_signature'] = None
        
        if 'Piano' in plan[measure]['instruments']:
            plan[measure]['key_signature'] = int(combined_data[measure]['Piano']['info']['ks_fifths'])
        elif plan[measure]['instruments']:
            key_signatures = [
                int(combined_data[measure][instrument]['info']['ks_fifths'])
                for instrument in plan[measure]['instruments'] 
                if not any(excluded in instrument for excluded in instruments_to_exclude)
            ]
            if key_signatures:
                plan[measure]['key_signature'] = max(set(key_signatures), key=key_signatures.count)
            else:
                plan[measure]['key_signature'] = None

        # --- 5. KEY MODE ---
        first_instrument = plan[measure]['instruments'][0] if plan[measure]['instruments'] else None
        if first_instrument and 'info' in combined_data[measure][first_instrument]:
            plan[measure]['key_mode'] = int(combined_data[measure][first_instrument]['info']['ks_mode'])
        else:
            plan[measure]['key_mode'] = None

        # --- 6. TEMPO ---
        if first_instrument and 'info' in combined_data[measure][first_instrument]:
            tempo_val = combined_data[measure][first_instrument]['info']['tempo']
            plan[measure]['tempo'] = int(tempo_val) if tempo_val is not None else None
        else:
            plan[measure]['tempo'] = None

        # --- 7. DYNAMICS (Added) ---
        # Retrieves the aggregated dynamics list (already sorted by time in combine_raw_xml_info)
        plan[measure]['dynamics'] = []
        if first_instrument and 'info' in combined_data[measure][first_instrument]:
            raw_dynamics = combined_data[measure][first_instrument]['info'].get('dynamics', [])
            # Format nicely and ensure standard types
            plan[measure]['dynamics'] = [
                # {'onset': int(d['rel_onset_div']), 'value': str(d['text'])} 
                str(d['text'])
                for d in raw_dynamics
            ]

        # --- 8. CHORDS ---
        plan[measure]['chords'] = []
        
        if not plan[measure]['instruments']:
            continue  

        onset_groups = {}

        for instrument in plan[measure]['instruments']:
            notes = combined_data[measure][instrument]['info']['notes']
            for note in notes:
                if note['is_grace'] == 0:
                    onset = note['rel_onset_div'] 
                    note_pitch = int(note['pitch'] % 12)

                    if onset not in onset_groups:
                        onset_groups[onset] = set()
                    
                    onset_groups[onset].add(note_pitch)

        unique_chords = []
        seen_chords = set()
        sorted_onsets = sorted(onset_groups.keys())

        for onset in sorted_onsets:
            current_chord_set = onset_groups[onset]
            
            if len(current_chord_set) < 2:
                continue

            chord_signature = tuple(sorted(current_chord_set))
            
            if chord_signature not in seen_chords:
                seen_chords.add(chord_signature)
                unique_chords.append(current_chord_set)

        plan[measure]['chords'] = unique_chords

    # Add all instruments to the plan
    plan['all_instruments'] = list(all_instruments)

    return plan

full_plan = get_full_plan(combined_data)
full_plan

{'Measure: 1': {'instruments': ['Piano'],
  'pitch_range': {'min': 60, 'max': 60},
  'note_density': 1.0,
  'time_signature': '2/2',
  'key_signature': -4,
  'key_mode': 1,
  'tempo': 232,
  'dynamics': ['p'],
  'chords': []},
 'Measure: 2': {'instruments': ['Piano'],
  'pitch_range': {'min': 65, 'max': 77},
  'note_density': 4.0,
  'time_signature': '2/2',
  'key_signature': -4,
  'key_mode': 1,
  'tempo': 232,
  'dynamics': ['crescendo'],
  'chords': []},
 'Measure: 3': {'instruments': ['Piano'],
  'pitch_range': {'min': 53, 'max': 80},
  'note_density': 14.0,
  'time_signature': '2/2',
  'key_signature': -4,
  'key_mode': 1,
  'tempo': 232,
  'dynamics': ['diminuendo'],
  'chords': [{0, 5, 8}]},
 'Measure: 4': {'instruments': ['Piano'],
  'pitch_range': {'min': 52, 'max': 79},
  'note_density': 8.0,
  'time_signature': '2/2',
  'key_signature': -4,
  'key_mode': 1,
  'tempo': 232,
  'dynamics': ['crescendo'],
  'chords': [{0, 4, 7, 10}]},
 'Measure: 5': {'instruments': ['Piano'],
  

In [ ]:
from text2music.data.utils.xml2plan import *

# xml_filepath = abc_filepath.replace('.abc', '.xml')
xml_filepath = "/home/ubuntu/keshav/text2score/text2music/notebooks/1203.xml"

data = get_raw_xml_info(xml_filepath)
combined_data = combine_raw_xml_info(data)
full_plan = get_full_plan(combined_data)
full_plan

In [7]:
plan = get_text_from_xml(xml_filepath)

print(plan)

Total Measures: 9

Instruments: Horn in F, Trombone 2, Tuba, Flute 2, Horn in F 2, B♭ Clarinet 4, Trombone, Chimes, Glockenspiel, B♭ Trumpet, Flute, B♭ Trumpet 2, Alto Saxophone, B♭ Clarinet 3, B♭ Clarinet, B♭ Clarinet 2, Timpani, Oboe

Measure: 1
  Instruments: Flute, Flute 2, B♭ Clarinet, B♭ Clarinet 2, B♭ Clarinet 3, B♭ Clarinet 4, Horn in F 2, Tuba, Chimes, Timpani
  Pitch Range: 38–81
  Note Density: 3.9
  Tempo: 76 BPM
  Time Signature: 3/4
  Chords: [9, 2], [2], [2, 6, 8, 9, 11]

Measure: 2
  Instruments: Flute, Flute 2, B♭ Clarinet, B♭ Clarinet 2, B♭ Clarinet 3, B♭ Clarinet 4, Oboe, Alto Saxophone, B♭ Trumpet 2, Horn in F, Trombone, Trombone 2, Chimes, Timpani
  Pitch Range: 50–86
  Note Density: 4.285714285714286
  Tempo: 76 BPM
  Time Signature: 3/4
  Chords: [2, 3, 4, 6, 8, 9, 11], [1, 2, 4, 6, 8, 9, 11], [1, 2, 3, 6, 8, 9, 11]

Measure: 15
  Instruments: Flute, Flute 2, B♭ Trumpet, Chimes
  Pitch Range: 62–75
  Note Density: 4.0
  Tempo: 76 BPM
  Time Signature: 3/4
  Chord

In [9]:
full_plan = get_full_plan_pipeline(xml_filepath)
full_plan

{'Measure: 1': {'instruments': ['Flute',
   'Flute 2',
   'B♭ Clarinet',
   'B♭ Clarinet 2',
   'B♭ Clarinet 3',
   'B♭ Clarinet 4',
   'Horn in F 2',
   'Tuba',
   'Chimes',
   'Timpani'],
  'pitch_range': {'min': np.int32(38), 'max': np.int32(81)},
  'note_density': 3.9,
  'time_signature': '3/4',
  'key_signature': 2,
  'key_mode': 1,
  'tempo': 76,
  'chords': [{np.int32(2), np.int32(9)},
   {np.int32(2)},
   {np.int32(2), np.int32(6), np.int32(8), np.int32(9), np.int32(11)}]},
 'Measure: 2': {'instruments': ['Flute',
   'Flute 2',
   'B♭ Clarinet',
   'B♭ Clarinet 2',
   'B♭ Clarinet 3',
   'B♭ Clarinet 4',
   'Oboe',
   'Alto Saxophone',
   'B♭ Trumpet 2',
   'Horn in F',
   'Trombone',
   'Trombone 2',
   'Chimes',
   'Timpani'],
  'pitch_range': {'min': np.int32(50), 'max': np.int32(86)},
  'note_density': 4.285714285714286,
  'time_signature': '3/4',
  'key_signature': 2,
  'key_mode': 1,
  'tempo': 76,
  'chords': [{np.int32(2),
    np.int32(3),
    np.int32(4),
    np.int32(

In [14]:
import pickle

plan_path = "/data/ABC_Dataset/SymphonyNet_Dataset_MXL_abci/outputs/1203.pkl"

with open(plan_path, 'rb') as f:
    plan = pickle.load(f)
plan

{'Measure: 1': {'instruments': ['Flute',
   'Flute 2',
   'B♭ Clarinet',
   'B♭ Clarinet 2',
   'B♭ Clarinet 3',
   'B♭ Clarinet 4',
   'Horn in F 2',
   'Tuba',
   'Chimes',
   'Timpani'],
  'pitch_range': {'min': 38, 'max': 81},
  'note_density': 3.9,
  'time_signature': '3/4',
  'key_signature': 2,
  'key_mode': 1,
  'tempo': 76,
  'chords': [[9, 2], [2], [2, 6, 8, 9, 11]]},
 'Measure: 2': {'instruments': ['Flute',
   'Flute 2',
   'B♭ Clarinet',
   'B♭ Clarinet 2',
   'B♭ Clarinet 3',
   'B♭ Clarinet 4',
   'Oboe',
   'Alto Saxophone',
   'B♭ Trumpet 2',
   'Horn in F',
   'Trombone',
   'Trombone 2',
   'Chimes',
   'Timpani'],
  'pitch_range': {'min': 50, 'max': 86},
  'note_density': 4.285714285714286,
  'time_signature': '3/4',
  'key_signature': 2,
  'key_mode': 1,
  'tempo': 76,
  'chords': [[2, 3, 4, 6, 8, 9, 11],
   [1, 2, 4, 6, 8, 9, 11],
   [1, 2, 3, 6, 8, 9, 11]]},
 'Measure: 3': {'instruments': ['Flute',
   'Flute 2',
   'B♭ Clarinet',
   'B♭ Clarinet 2',
   'B♭ Clarine